In [1]:
# ============================================================
# ML PHASE - SETUP
# ============================================================

import pandas as pd
import numpy as np

# ----------------------------
# Load datasets
# ----------------------------

employees = pd.read_csv(
    "../data/raw/employees.csv"
)

rooms = pd.read_csv(
    "../data/raw/rooms.csv"
)

room_booking_requests = pd.read_csv(
    "../data/raw/room_booking_requests.csv",
    parse_dates=[
        "requested_start_datetime",
        "requested_end_datetime"
    ]
)

room_bookings = pd.read_csv(
    "../data/raw/room_bookings.csv",
    parse_dates=[
        "requested_start_datetime",
        "requested_end_datetime",
        "allocated_start_datetime",
        "allocated_end_datetime"
    ]
)

# ----------------------------
# Basic verification
# ----------------------------

print("Employees:", employees.shape)
print("Rooms:", rooms.shape)
print("Booking requests:", room_booking_requests.shape)
print("Room bookings:", room_bookings.shape)

print("\nRoom booking columns:")
print(room_bookings.columns.tolist())

print("\nAll datasets loaded successfully.")

Employees: (1000, 10)
Rooms: (50, 5)
Booking requests: (40000, 10)
Room bookings: (40000, 19)

Room booking columns:
['booking_id', 'employee_id', 'booking_date', 'requested_start_datetime', 'allocated_start_datetime', 'requested_end_datetime', 'allocated_end_datetime', 'requested_duration_minutes', 'max_time_flexibility_minutes', 'requested_room_type', 'allocated_room_id', 'allocated_room_type', 'requested_floor', 'allocated_floor', 'requested_capacity', 'room_capacity', 'time_relaxation_minutes', 'allocation_rule', 'booking_status']

All datasets loaded successfully.


In [2]:
print("Booking columns:")
print(room_bookings.columns.tolist())

print("\nRequest columns:")
print(room_booking_requests.columns.tolist())

print("\nRoom columns:")
print(rooms.columns.tolist())

Booking columns:
['booking_id', 'employee_id', 'booking_date', 'requested_start_datetime', 'allocated_start_datetime', 'requested_end_datetime', 'allocated_end_datetime', 'requested_duration_minutes', 'max_time_flexibility_minutes', 'requested_room_type', 'allocated_room_id', 'allocated_room_type', 'requested_floor', 'allocated_floor', 'requested_capacity', 'room_capacity', 'time_relaxation_minutes', 'allocation_rule', 'booking_status']

Request columns:
['booking_id', 'employee_id', 'booking_date', 'requested_start_datetime', 'requested_end_datetime', 'requested_duration_minutes', 'requested_room_type', 'requested_capacity', 'max_time_flexibility_minutes', 'requested_floor']

Room columns:
['room_id', 'floor', 'room_type', 'room_size', 'capacity']


In [3]:
allocation_columns = [
    col for col in room_bookings.columns
    if (
        "alloc" in col.lower()
        or "room" in col.lower()
        or "status" in col.lower()
    )
]

print(allocation_columns)

['allocated_start_datetime', 'allocated_end_datetime', 'requested_room_type', 'allocated_room_id', 'allocated_room_type', 'allocated_floor', 'room_capacity', 'allocation_rule', 'booking_status']


In [4]:
print("Booking status:")
print(room_bookings["booking_status"].value_counts(dropna=False))

print("\nAllocation rules:")
print(room_bookings["allocation_rule"].value_counts(dropna=False))

print("\nAllocated room IDs missing:")
print(room_bookings["allocated_room_id"].isna().sum())

print("\nAllocated room IDs present:")
print(room_bookings["allocated_room_id"].notna().sum())

Booking status:
booking_status
Allocated    32442
Skipped       7558
Name: count, dtype: int64

Allocation rules:
allocation_rule
exact_type_exact_floor       8219
skipped_no_room_available    7558
similar_type_any_floor       7309
similar_type_exact_floor     6241
exact_type_nearby_floor      5592
exact_type_any_floor         5081
Name: count, dtype: int64

Allocated room IDs missing:
7558

Allocated room IDs present:
32442


In [5]:
allocated_bookings = room_bookings[
    room_bookings["booking_status"] == "Allocated"
].copy()

allocated_room_check = allocated_bookings.merge(
    rooms[["room_id", "floor", "room_type", "capacity"]],
    left_on="allocated_room_id",
    right_on="room_id",
    how="left",
    indicator=True
)

print("Allocated bookings:", len(allocated_bookings))
print("Matched room records:", (allocated_room_check["_merge"] == "both").sum())
print("Missing room records:", (allocated_room_check["_merge"] == "left_only").sum())

Allocated bookings: 32442
Matched room records: 32442
Missing room records: 0


In [6]:
allocated_requests = (
    room_booking_requests
    .merge(
        room_bookings[
            [
                "booking_id",
                "allocated_room_id",
                "allocated_start_datetime",
                "allocated_end_datetime",
                "allocated_floor",
                "allocated_room_type",
                "room_capacity",
                "allocation_rule",
                "booking_status"
            ]
        ],
        on="booking_id",
        how="inner"
    )
)

allocated_requests = allocated_requests[
    allocated_requests["booking_status"] == "Allocated"
].copy()

print("Allocated request rows:", len(allocated_requests))
print("Unique bookings:", allocated_requests["booking_id"].nunique())
print("Unique employees:", allocated_requests["employee_id"].nunique())

allocated_requests.head()

Allocated request rows: 32442
Unique bookings: 32442
Unique employees: 998


,booking_id,employee_id,booking_date,requested_start_datetime,requested_end_datetime,requested_duration_minutes,requested_room_type,requested_capacity,max_time_flexibility_minutes,requested_floor,allocated_room_id,allocated_start_datetime,allocated_end_datetime,allocated_floor,allocated_room_type,room_capacity,allocation_rule,booking_status
0,RB06430,E0742,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Meeting,16,60,3,R008,2025-01-01 08:00:00,2025-01-01 09:15:00,1.0,Meeting,16.0,exact_type_any_floor,Allocated
1,RB10999,E0166,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Conference,4,60,5,R042,2025-01-01 08:00:00,2025-01-01 09:15:00,5.0,Conference,4.0,exact_type_exact_floor,Allocated
2,RB19847,E0605,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Conference,12,0,1,R034,2025-01-01 08:00:00,2025-01-01 09:15:00,4.0,Conference,16.0,exact_type_any_floor,Allocated
3,RB22582,E0904,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:15:00,75,Meeting,4,30,3,R024,2025-01-01 08:00:00,2025-01-01 09:15:00,3.0,Meeting,8.0,exact_type_exact_floor,Allocated
4,RB25100,E0378,2025-01-01,2025-01-01 08:00:00,2025-01-01 09:00:00,60,Meeting,10,0,5,R041,2025-01-01 08:00:00,2025-01-01 09:00:00,5.0,Focus,16.0,similar_type_exact_floor,Allocated


In [7]:
room_intervals = (
    room_bookings[
        room_bookings["booking_status"] == "Allocated"
    ][
        [
            "allocated_room_id",
            "allocated_start_datetime",
            "allocated_end_datetime"
        ]
    ]
    .dropna()
    .sort_values(
        ["allocated_room_id", "allocated_start_datetime"]
    )
    .reset_index(drop=True)
)

print("Allocated intervals:", len(room_intervals))
print("Unique rooms with bookings:", room_intervals["allocated_room_id"].nunique())

room_intervals.head()

Allocated intervals: 32442
Unique rooms with bookings: 50


,allocated_room_id,allocated_start_datetime,allocated_end_datetime
0,R001,2025-01-01 08:15:00,2025-01-01 09:15:00
1,R001,2025-01-01 09:15:00,2025-01-01 10:15:00
2,R001,2025-01-01 10:15:00,2025-01-01 11:00:00
3,R001,2025-01-01 11:15:00,2025-01-01 12:15:00
4,R001,2025-01-02 08:00:00,2025-01-02 09:30:00


In [ ]:
# ================================================================================================================
#                                              CANDIDATE ROOMS AVAILABILITY CHECKER (DEMO CHECKING)
# ================================================================================================================


def is_room_available(room_id, start_time, end_time):
    room_bookings_for_room = room_intervals[
        room_intervals["allocated_room_id"] == room_id
    ]

    overlaps = (
        (room_bookings_for_room["allocated_start_datetime"] < end_time)
        &
        (room_bookings_for_room["allocated_end_datetime"] > start_time)
    )

    return not overlaps.any()


# Quick test
test_room = room_intervals.iloc[0]["allocated_room_id"]
test_start = room_intervals.iloc[0]["allocated_start_datetime"]
test_end = room_intervals.iloc[0]["allocated_end_datetime"]

print("Room:", test_room)
print("Available:", is_room_available(test_room, test_start, test_end))

Room: R001
Available: False


In [ ]:
# ================================================================================================================
#                     USING BISECT SO THAT NO UNNECESSARY TIME WASTE IN CHECKING ALL HISTORY FROM STARTING
# ================================================================================================================


from bisect import bisect_left

# Store each room's booking intervals separately
room_interval_lookup = {}

for room_id, group in room_intervals.groupby("allocated_room_id"):
    starts = group["allocated_start_datetime"].tolist()
    ends = group["allocated_end_datetime"].tolist()

    room_interval_lookup[room_id] = {
        "starts": starts,
        "ends": ends
    }


def is_room_available_fast(room_id, start_time, end_time):
    if room_id not in room_interval_lookup:
        return True

    starts = room_interval_lookup[room_id]["starts"]
    ends = room_interval_lookup[room_id]["ends"]

    # Find where the requested start time would be inserted
    idx = bisect_left(starts, start_time)

    # Check the interval immediately before the insertion point
    if idx > 0 and ends[idx - 1] > start_time:
        return False

    # Check the interval at the insertion point
    if idx < len(starts) and starts[idx] < end_time:
        return False

    return True


# Verify against the same occupied interval we tested earlier
test_room = room_intervals.iloc[0]["allocated_room_id"]
test_start = room_intervals.iloc[0]["allocated_start_datetime"]
test_end = room_intervals.iloc[0]["allocated_end_datetime"]

print("Room:", test_room)
print("Available:", is_room_available_fast(test_room, test_start, test_end))

Room: R001
Available: False


In [22]:
# ================================================================================================================
#             ALSO CHECKING FOR {MAX_FLEXIBILTY_MIN}, GENERATING TIMESTAMPS FOR DUMMY {60-MINUTES}
# ================================================================================================================

def generate_allowed_start_times(requested_start, flexibility_minutes):
    allowed_times = []

    for offset in range(0, flexibility_minutes + 1, 15):
        if offset == 0:
            allowed_times.append(requested_start)
        else:
            allowed_times.append(
                requested_start - pd.Timedelta(minutes=offset)
            )
            allowed_times.append(
                requested_start + pd.Timedelta(minutes=offset)
            )

    return sorted(allowed_times)

In [23]:
# ================================================================================================================
#             CHECKING KITNE REQUESTS MAX_FLEXIBILITY_MIN KE ANDAR HI ARHE HAIN 
# ================================================================================================================

allocated_requests["start_shift_minutes"] = (
    (
        allocated_requests["allocated_start_datetime"]
        - allocated_requests["requested_start_datetime"]
    ).dt.total_seconds() / 60
)

allocated_requests["within_flexibility"] = (
    allocated_requests["start_shift_minutes"].abs()
    <= allocated_requests["max_time_flexibility_minutes"]
)

print(allocated_requests["within_flexibility"].value_counts())
print(
    "\nMaximum absolute start-time shift:",
    allocated_requests["start_shift_minutes"].abs().max(),
    "minutes"
)

within_flexibility
True    32442
Name: count, dtype: int64

Maximum absolute start-time shift: 105.0 minutes


In [24]:
flexibility_violations = allocated_requests[
    ~allocated_requests["within_flexibility"]
][
    [
        "booking_id",
        "employee_id",
        "requested_start_datetime",
        "allocated_start_datetime",
        "max_time_flexibility_minutes",
        "start_shift_minutes",
        "requested_room_type",
        "allocated_room_type",
        "requested_floor",
        "allocated_floor",
        "allocation_rule"
    ]
].copy()

print("Number of violations:", len(flexibility_violations))
print("\nViolation by allocation rule:")
print(flexibility_violations["allocation_rule"].value_counts())

print("\nViolations:")
display(flexibility_violations.head(20))

Number of violations: 0

Violation by allocation rule:
Series([], Name: count, dtype: int64)

Violations:


,booking_id,employee_id,requested_start_datetime,allocated_start_datetime,max_time_flexibility_minutes,start_shift_minutes,requested_room_type,allocated_room_type,requested_floor,allocated_floor,allocation_rule


In [25]:
from bisect import bisect_left

room_intervals = (
    room_bookings[room_bookings["booking_status"] == "Allocated"][
        [
            "booking_id",
            "allocated_room_id",
            "allocated_start_datetime",
            "allocated_end_datetime"
        ]
    ]
    .dropna()
    .sort_values(["allocated_room_id", "allocated_start_datetime"])
    .reset_index(drop=True)
)

room_interval_lookup = {}

for room_id, group in room_intervals.groupby("allocated_room_id"):
    room_interval_lookup[room_id] = {
        "starts": group["allocated_start_datetime"].tolist(),
        "ends": group["allocated_end_datetime"].tolist(),
        "booking_ids": group["booking_id"].tolist()
    }

def is_room_available_fast(
    room_id,
    start_time,
    end_time,
    exclude_booking_id=None
):
    if room_id not in room_interval_lookup:
        return True

    starts = room_interval_lookup[room_id]["starts"]
    ends = room_interval_lookup[room_id]["ends"]
    booking_ids = room_interval_lookup[room_id]["booking_ids"]

    idx = bisect_left(starts, start_time)

    # Check interval immediately before candidate
    if idx > 0:
        previous_idx = idx - 1

        if (
            ends[previous_idx] > start_time
            and booking_ids[previous_idx] != exclude_booking_id
        ):
            return False

    # Check interval at/after candidate start
    while idx < len(starts) and starts[idx] < end_time:

        if booking_ids[idx] != exclude_booking_id:
            return False

        idx += 1

    return True

In [26]:
# ================================================================================================================
#                        GENERATING CANDIDATES     (FOR 200-REQUESTS ONLY)
# ================================================================================================================


def generate_candidates_for_request(request, rooms_df):
    candidates = []

    booking_id = request["booking_id"]
    requested_start = request["requested_start_datetime"]
    requested_end = request["requested_end_datetime"]
    duration = requested_end - requested_start
    max_flexibility = int(request["max_time_flexibility_minutes"])

    allocated_room_id = request["allocated_room_id"]
    allocated_start = request["allocated_start_datetime"]

    allowed_start_times = generate_allowed_start_times(
        requested_start,
        max_flexibility
    )

    # Rooms must satisfy hard capacity constraint
    eligible_rooms = rooms_df[
        rooms_df["capacity"] >= request["requested_capacity"]
    ]

    for _, room in eligible_rooms.iterrows():

        for candidate_start in allowed_start_times:

            candidate_end = candidate_start + duration

            if is_room_available_fast(
                room["room_id"],
                candidate_start,
                candidate_end,
                exclude_booking_id=booking_id
            ):

                is_positive = (
                    room["room_id"] == allocated_room_id
                    and candidate_start == allocated_start
                )

                candidates.append({
                    "booking_id": booking_id,
                    "employee_id": request["employee_id"],
                    "candidate_room_id": room["room_id"],
                    "candidate_start_datetime": candidate_start,
                    "candidate_end_datetime": candidate_end,
                    "candidate_floor": room["floor"],
                    "candidate_room_type": room["room_type"],
                    "candidate_capacity": room["capacity"],
                    "target": int(is_positive)
                })

    return candidates


# Test on first 200 allocated requests
test_requests = allocated_requests.head(200)

test_candidates = []

for _, request in test_requests.iterrows():
    test_candidates.extend(
        generate_candidates_for_request(request, rooms)
    )

test_candidates = pd.DataFrame(test_candidates)

print("Test requests:", len(test_requests))
print("Generated candidates:", len(test_candidates))
print("\nTarget distribution:")
print(test_candidates["target"].value_counts())

print("\nPositive candidates:", test_candidates["target"].sum())
print(
    "Requests with positive candidate:",
    test_candidates.groupby("booking_id")["target"].sum().gt(0).sum()
)

Test requests: 200
Generated candidates: 7784

Target distribution:
target
0    7584
1     200
Name: count, dtype: int64

Positive candidates: 200
Requests with positive candidate: 200


In [19]:
positive_counts = (
    test_candidates.groupby("booking_id")["target"]
    .sum()
)

missing_booking_ids = positive_counts[
    positive_counts == 0
].index.tolist()

print("Missing positive booking IDs:", missing_booking_ids)

print("\nDetails of missing request:")

display(
    test_requests[
        test_requests["booking_id"].isin(missing_booking_ids)
    ][
        [
            "booking_id",
            "employee_id",
            "requested_start_datetime",
            "requested_end_datetime",
            "requested_room_type",
            "requested_floor",
            "requested_capacity",
            "max_time_flexibility_minutes",
            "allocated_room_id",
            "allocated_start_datetime",
            "allocated_end_datetime",
            "allocated_room_type",
            "allocated_floor",
            "room_capacity",
            "allocation_rule"
        ]
    ]
)

Missing positive booking IDs: []

Details of missing request:


,booking_id,employee_id,requested_start_datetime,requested_end_datetime,requested_room_type,requested_floor,requested_capacity,max_time_flexibility_minutes,allocated_room_id,allocated_start_datetime,allocated_end_datetime,allocated_room_type,allocated_floor,room_capacity,allocation_rule


In [20]:
print("Missing booking IDs:", missing_booking_ids)

if missing_booking_ids:
    missing_request = test_requests[
        test_requests["booking_id"].isin(missing_booking_ids)
    ]

    print("\nMissing request:")
    print(missing_request[
        [
            "booking_id",
            "employee_id",
            "requested_start_datetime",
            "requested_end_datetime",
            "requested_room_type",
            "requested_floor",
            "requested_capacity",
            "max_time_flexibility_minutes",
            "allocated_room_id",
            "allocated_start_datetime",
            "allocated_end_datetime",
            "allocated_room_type",
            "allocated_floor",
            "room_capacity",
            "allocation_rule"
        ]
    ].to_string(index=False))
else:
    print("\nNo missing booking found.")

Missing booking IDs: []

No missing booking found.


In [21]:
test_booking_ids = set(test_requests["booking_id"])
candidate_booking_ids = set(test_candidates["booking_id"])

zero_candidate_booking_ids = sorted(
    test_booking_ids - candidate_booking_ids
)

print("Requests with ZERO candidates:", len(zero_candidate_booking_ids))
print("Booking IDs:", zero_candidate_booking_ids)

if zero_candidate_booking_ids:
    print("\nDetails:")
    display(
        test_requests[
            test_requests["booking_id"].isin(zero_candidate_booking_ids)
        ][
            [
                "booking_id",
                "employee_id",
                "requested_start_datetime",
                "requested_end_datetime",
                "requested_room_type",
                "requested_floor",
                "requested_capacity",
                "max_time_flexibility_minutes",
                "allocated_room_id",
                "allocated_start_datetime",
                "allocated_end_datetime",
                "allocated_room_type",
                "allocated_floor",
                "room_capacity",
                "allocation_rule"
            ]
        ]
    )

Requests with ZERO candidates: 1
Booking IDs: ['RB31383']

Details:


,booking_id,employee_id,requested_start_datetime,requested_end_datetime,requested_room_type,requested_floor,requested_capacity,max_time_flexibility_minutes,allocated_room_id,allocated_start_datetime,allocated_end_datetime,allocated_room_type,allocated_floor,room_capacity,allocation_rule
20,RB31383,E0294,2025-01-01 08:15:00,2025-01-01 09:15:00,Conference,2,10,60,R018,2025-01-01 07:15:00,2025-01-01 08:15:00,Training,2.0,16.0,similar_type_exact_floor


In [27]:
positive_candidates = test_candidates[
    test_candidates["target"] == 1
].copy()

positive_counts = (
    positive_candidates
    .groupby("booking_id")
    .size()
)

print("Total positive candidates:", len(positive_candidates))
print("Unique bookings with positive:", positive_counts.size)

print("\nPositive candidates per booking:")
print(positive_counts.value_counts().sort_index())

print(
    "\nBookings with exactly one positive:",
    (positive_counts == 1).sum()
)

print(
    "Bookings with multiple positives:",
    (positive_counts > 1).sum()
)

print(
    "Bookings with zero positives:",
    len(test_requests) - positive_counts.size
)

Total positive candidates: 200
Unique bookings with positive: 200

Positive candidates per booking:
1    200
Name: count, dtype: int64

Bookings with exactly one positive: 200
Bookings with multiple positives: 0
Bookings with zero positives: 0


In [28]:
full_candidates = []

total_requests = len(allocated_requests)

for i, (_, request) in enumerate(allocated_requests.iterrows(), start=1):

    full_candidates.extend(
        generate_candidates_for_request(request, rooms)
    )

    if i % 2000 == 0 or i == total_requests:
        print(
            f"Processed {i:,}/{total_requests:,} requests | "
            f"Candidates generated: {len(full_candidates):,}"
        )

candidate_dataset = pd.DataFrame(full_candidates)

print("\n========== FINAL CANDIDATE DATASET ==========")
print("Rows:", len(candidate_dataset))
print("Columns:", candidate_dataset.shape[1])
print("\nTarget distribution:")
print(candidate_dataset["target"].value_counts())

print("\nUnique bookings:")
print(candidate_dataset["booking_id"].nunique())

print("\nPositive candidates:")
print(candidate_dataset["target"].sum())

Processed 2,000/32,442 requests | Candidates generated: 69,829
Processed 4,000/32,442 requests | Candidates generated: 133,693
Processed 6,000/32,442 requests | Candidates generated: 191,691
Processed 8,000/32,442 requests | Candidates generated: 254,170
Processed 10,000/32,442 requests | Candidates generated: 316,532
Processed 12,000/32,442 requests | Candidates generated: 375,126
Processed 14,000/32,442 requests | Candidates generated: 438,656
Processed 16,000/32,442 requests | Candidates generated: 500,304
Processed 18,000/32,442 requests | Candidates generated: 563,752
Processed 20,000/32,442 requests | Candidates generated: 624,427
Processed 22,000/32,442 requests | Candidates generated: 691,076
Processed 24,000/32,442 requests | Candidates generated: 752,959
Processed 26,000/32,442 requests | Candidates generated: 810,877
Processed 28,000/32,442 requests | Candidates generated: 870,515
Processed 30,000/32,442 requests | Candidates generated: 925,933
Processed 32,000/32,442 reques

In [29]:
# Sort all historical requests chronologically
all_requests = room_booking_requests.copy()

all_requests["requested_start_hour"] = (
    all_requests["requested_start_datetime"].dt.hour
)

all_requests = all_requests.sort_values(
    ["employee_id", "requested_start_datetime", "booking_id"]
).reset_index(drop=True)


# Columns whose historical behaviour we want to track
behavior_columns = {
    "requested_room_type": ["Conference", "Focus", "Meeting", "Training"],
    "requested_floor": [1, 2, 3, 4, 5],
    "requested_start_hour": [8, 9, 10, 11],
    "requested_capacity": [2, 4, 6, 8, 10, 12, 16]
}


def calculate_distribution(history, column, values):
    counts = history[column].value_counts()
    total = len(history)

    if total == 0:
        return {
            f"{column}_{value}_pct": 0.0
            for value in values
        }

    return {
        f"{column}_{value}_pct": (
            counts.get(value, 0) / total * 100
        )
        for value in values
    }


behavior_rows = []

for _, request in allocated_requests.iterrows():

    employee_id = request["employee_id"]
    current_time = request["requested_start_datetime"]

    employee_history = all_requests[
        (all_requests["employee_id"] == employee_id) &
        (all_requests["requested_start_datetime"] < current_time)
    ]

    # -----------------------------
    # Long-term behaviour
    # -----------------------------
    long_term = {}

    for column, values in behavior_columns.items():
        long_term.update(
            calculate_distribution(
                employee_history,
                column,
                values
            )
        )

    # Duration
    if len(employee_history) > 0:
        long_term["duration_mean"] = (
            employee_history["requested_duration_minutes"].mean()
        )
        long_term["duration_std"] = (
            employee_history["requested_duration_minutes"].std()
            if len(employee_history) > 1 else 0.0
        )

        long_term["flexibility_mean"] = (
            employee_history["max_time_flexibility_minutes"].mean()
        )
        long_term["flexibility_std"] = (
            employee_history["max_time_flexibility_minutes"].std()
            if len(employee_history) > 1 else 0.0
        )
    else:
        long_term["duration_mean"] = 0.0
        long_term["duration_std"] = 0.0
        long_term["flexibility_mean"] = 0.0
        long_term["flexibility_std"] = 0.0


    # -----------------------------
    # Recent 30-day behaviour
    # -----------------------------
    recent_start = current_time - pd.Timedelta(days=30)

    recent_history = employee_history[
        employee_history["requested_start_datetime"] >= recent_start
    ]

    recent = {}

    for column in [
        "requested_room_type",
        "requested_floor",
        "requested_start_hour",
        "requested_capacity"
    ]:
        recent.update(
            calculate_distribution(
                recent_history,
                column,
                behavior_columns[column]
            )
        )


    # -----------------------------
    # Behaviour shifts
    # -----------------------------
    shifts = {}

    for column, values in behavior_columns.items():

        long_values = [
            long_term[f"{column}_{value}_pct"]
            for value in values
        ]

        recent_values = [
            recent[f"{column}_{value}_pct"]
            for value in values
        ]

        shifts[f"{column}_shift_pct"] = (
            sum(
                abs(long_value - recent_value)
                for long_value, recent_value
                in zip(long_values, recent_values)
            ) / 2
        )


    behavior_rows.append({
        "booking_id": request["booking_id"],
        "employee_id": employee_id,
        **long_term,
        **recent,
        **shifts
    })


request_behavior_features = pd.DataFrame(behavior_rows)

print("Behavior feature rows:", len(request_behavior_features))
print("Behavior feature columns:", request_behavior_features.shape[1])

print("\nFirst 10 columns:")
print(request_behavior_features.columns.tolist()[:10])

print("\nSample:")
display(request_behavior_features.head())

Behavior feature rows: 32442
Behavior feature columns: 30

First 10 columns:
['booking_id', 'employee_id', 'requested_room_type_Conference_pct', 'requested_room_type_Focus_pct', 'requested_room_type_Meeting_pct', 'requested_room_type_Training_pct', 'requested_floor_1_pct', 'requested_floor_2_pct', 'requested_floor_3_pct', 'requested_floor_4_pct']

Sample:


,booking_id,employee_id,requested_room_type_Conference_pct,requested_room_type_Focus_pct,requested_room_type_Meeting_pct,requested_room_type_Training_pct,requested_floor_1_pct,requested_floor_2_pct,requested_floor_3_pct,requested_floor_4_pct,...,requested_capacity_12_pct,requested_capacity_16_pct,duration_mean,duration_std,flexibility_mean,flexibility_std,requested_room_type_shift_pct,requested_floor_shift_pct,requested_start_hour_shift_pct,requested_capacity_shift_pct
0,RB06430,E0742,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,RB10999,E0166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,RB19847,E0605,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,RB22582,E0904,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,RB25100,E0378,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# =========================================================================================================
#       CHECKING LEAKAGE SAFE BEHAVIOUR
# =========================================================================================================

history_counts = []

for _, request in allocated_requests.iterrows():

    employee_id = request["employee_id"]
    current_time = request["requested_start_datetime"]

    previous_requests = all_requests[
        (all_requests["employee_id"] == employee_id) &
        (all_requests["requested_start_datetime"] < current_time)
    ]

    history_counts.append(len(previous_requests))

history_counts = pd.Series(history_counts)

print("Total requests:", len(history_counts))

print(
    "\nRequests with NO previous history:",
    (history_counts == 0).sum()
)

print(
    "Requests with previous history:",
    (history_counts > 0).sum()
)

print(
    "\nAverage previous requests:",
    round(history_counts.mean(), 2)
)

print(
    "Maximum previous requests:",
    history_counts.max()
)

print(
    "\nNaN values in behaviour features:",
    request_behavior_features.isna().sum().sum()
)

Total requests: 32442

Requests with NO previous history: 837
Requests with previous history: 31605

Average previous requests: 24.24
Maximum previous requests: 90

NaN values in behaviour features: 0


In [31]:
ml_dataset = candidate_dataset.merge(
    request_behavior_features,
    on=["booking_id", "employee_id"],
    how="left"
)

print("ML dataset shape:", ml_dataset.shape)

print("\nMissing values:")
print(ml_dataset.isna().sum().sum())

print("\nTarget distribution:")
print(ml_dataset["target"].value_counts())

print("\nSample:")
display(ml_dataset.head())

ML dataset shape: (999927, 37)

Missing values:
0

Target distribution:
target
0    967485
1     32442
Name: count, dtype: int64

Sample:


,booking_id,employee_id,candidate_room_id,candidate_start_datetime,candidate_end_datetime,candidate_floor,candidate_room_type,candidate_capacity,target,requested_room_type_Conference_pct,...,requested_capacity_12_pct,requested_capacity_16_pct,duration_mean,duration_std,flexibility_mean,flexibility_std,requested_room_type_shift_pct,requested_floor_shift_pct,requested_start_hour_shift_pct,requested_capacity_shift_pct
0,RB06430,E0742,R005,2025-01-01 07:00:00,2025-01-01 08:15:00,1,Training,16,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,RB06430,E0742,R006,2025-01-01 07:00:00,2025-01-01 08:15:00,1,Focus,16,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,RB06430,E0742,R008,2025-01-01 07:00:00,2025-01-01 08:15:00,1,Meeting,16,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,RB06430,E0742,R008,2025-01-01 07:15:00,2025-01-01 08:30:00,1,Meeting,16,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,RB06430,E0742,R008,2025-01-01 07:30:00,2025-01-01 08:45:00,1,Meeting,16,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
request_features = allocated_requests[
    [
        "booking_id",
        "requested_room_type",
        "requested_floor",
        "requested_capacity",
        "requested_start_datetime",
        "requested_duration_minutes",
        "max_time_flexibility_minutes"
    ]
].copy()

request_features["requested_start_hour"] = (
    request_features["requested_start_datetime"].dt.hour
)

request_features["requested_day_of_week"] = (
    request_features["requested_start_datetime"].dt.dayofweek
)

request_features = request_features.drop(
    columns=["requested_start_datetime"]
)

ml_dataset = ml_dataset.merge(
    request_features,
    on="booking_id",
    how="left"
)

print("ML dataset shape:", ml_dataset.shape)
print("Missing values:", ml_dataset.isna().sum().sum())

print("\nNew request features:")
print([
    col for col in request_features.columns
    if col != "booking_id"
])

ML dataset shape: (999927, 44)
Missing values: 0

New request features:
['requested_room_type', 'requested_floor', 'requested_capacity', 'requested_duration_minutes', 'max_time_flexibility_minutes', 'requested_start_hour', 'requested_day_of_week']


In [35]:
ml_dataset = ml_dataset.merge(
    allocated_requests[
        ["booking_id", "requested_start_datetime"]
    ],
    on="booking_id",
    how="left"
)

print("Requested datetime restored:", "requested_start_datetime" in ml_dataset.columns)
print("Missing requested datetime:", ml_dataset["requested_start_datetime"].isna().sum())

Requested datetime restored: True
Missing requested datetime: 0


In [36]:
# Room-type relationship
ml_dataset["room_type_match"] = (
    ml_dataset["candidate_room_type"]
    == ml_dataset["requested_room_type"]
).astype(int)


# Floor relationship
ml_dataset["floor_difference"] = (
    ml_dataset["candidate_floor"]
    - ml_dataset["requested_floor"]
).abs()


# Capacity relationship
ml_dataset["capacity_difference"] = (
    ml_dataset["candidate_capacity"]
    - ml_dataset["requested_capacity"]
).abs()


# Candidate start-time relationship
ml_dataset["candidate_start_hour"] = (
    ml_dataset["candidate_start_datetime"].dt.hour
)

ml_dataset["start_time_difference_minutes"] = (
    (
        ml_dataset["candidate_start_datetime"]
        - ml_dataset["requested_start_datetime"]
    )
    .dt.total_seconds()
    .abs()
    / 60
)


# Candidate time flexibility usage
ml_dataset["flexibility_used_pct"] = np.where(
    ml_dataset["max_time_flexibility_minutes"] > 0,
    (
        ml_dataset["start_time_difference_minutes"]
        / ml_dataset["max_time_flexibility_minutes"]
    ) * 100,
    0
)


print("ML dataset shape:", ml_dataset.shape)
print("Missing values:", ml_dataset.isna().sum().sum())

print("\nRelationship features added:")
print([
    "room_type_match",
    "floor_difference",
    "capacity_difference",
    "candidate_start_hour",
    "start_time_difference_minutes",
    "flexibility_used_pct"
])

ML dataset shape: (999927, 51)
Missing values: 0

Relationship features added:
['room_type_match', 'floor_difference', 'capacity_difference', 'candidate_start_hour', 'start_time_difference_minutes', 'flexibility_used_pct']


In [37]:
# Get one request date for each booking
booking_dates = (
    ml_dataset[
        ["booking_id", "requested_start_datetime"]
    ]
    .drop_duplicates("booking_id")
    .sort_values("requested_start_datetime")
    .reset_index(drop=True)
)

# 80% of bookings for training, 20% for testing
split_index = int(len(booking_dates) * 0.80)

train_booking_ids = set(
    booking_dates.iloc[:split_index]["booking_id"]
)

test_booking_ids = set(
    booking_dates.iloc[split_index:]["booking_id"]
)

train_data = ml_dataset[
    ml_dataset["booking_id"].isin(train_booking_ids)
].copy()

test_data = ml_dataset[
    ml_dataset["booking_id"].isin(test_booking_ids)
].copy()

print("Training requests:", train_data["booking_id"].nunique())
print("Testing requests:", test_data["booking_id"].nunique())

print("\nTraining candidate rows:", len(train_data))
print("Testing candidate rows:", len(test_data))

print("\nTraining date range:")
print(
    train_data["requested_start_datetime"].min(),
    "to",
    train_data["requested_start_datetime"].max()
)

print("\nTesting date range:")
print(
    test_data["requested_start_datetime"].min(),
    "to",
    test_data["requested_start_datetime"].max()
)

print("\nTrain positives:", train_data["target"].sum())
print("Test positives:", test_data["target"].sum())

Training requests: 25953
Testing requests: 6489

Training candidate rows: 809515
Testing candidate rows: 190412

Training date range:
2025-01-01 08:00:00 to 2025-10-20 10:00:00

Testing date range:
2025-10-20 10:00:00 to 2025-12-31 11:30:00

Train positives: 25953
Test positives: 6489


In [38]:
# Columns that identify a row/request but should NOT be model features
id_columns = [
    "booking_id",
    "employee_id",
    "candidate_room_id"
]

# Datetime columns are useful for splitting/analysis,
# but raw timestamps should not be directly fed to the model
datetime_columns = [
    "requested_start_datetime",
    "candidate_start_datetime",
    "candidate_end_datetime"
]

# Target
y_train = train_data["target"].copy()
y_test = test_data["target"].copy()

# Feature columns
feature_columns = [
    col for col in train_data.columns
    if col not in id_columns + datetime_columns + ["target"]
]

X_train = train_data[feature_columns].copy()
X_test = test_data[feature_columns].copy()

print("Number of features:", len(feature_columns))

print("\nFeature columns:")
for i, col in enumerate(feature_columns, start=1):
    print(f"{i}. {col}")

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTarget shapes:")
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Number of features: 44

Feature columns:
1. candidate_floor
2. candidate_room_type
3. candidate_capacity
4. requested_room_type_Conference_pct
5. requested_room_type_Focus_pct
6. requested_room_type_Meeting_pct
7. requested_room_type_Training_pct
8. requested_floor_1_pct
9. requested_floor_2_pct
10. requested_floor_3_pct
11. requested_floor_4_pct
12. requested_floor_5_pct
13. requested_start_hour_8_pct
14. requested_start_hour_9_pct
15. requested_start_hour_10_pct
16. requested_start_hour_11_pct
17. requested_capacity_2_pct
18. requested_capacity_4_pct
19. requested_capacity_6_pct
20. requested_capacity_8_pct
21. requested_capacity_10_pct
22. requested_capacity_12_pct
23. requested_capacity_16_pct
24. duration_mean
25. duration_std
26. flexibility_mean
27. flexibility_std
28. requested_room_type_shift_pct
29. requested_floor_shift_pct
30. requested_start_hour_shift_pct
31. requested_capacity_shift_pct
32. requested_room_type
33. requested_floor
34. requested_capacity
35. requested_dura

In [41]:
#  ENCODING

from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "candidate_room_type",
    "requested_room_type"
]

# Fit encoder ONLY on training data
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_encoded = encoder.transform(
    X_test[categorical_features]
)

# Remove categorical columns from original numeric data
X_train_numeric = X_train.drop(
    columns=categorical_features
).reset_index(drop=True)

X_test_numeric = X_test.drop(
    columns=categorical_features
).reset_index(drop=True)

# Convert encoded arrays into DataFrames
encoded_columns = encoder.get_feature_names_out(
    categorical_features
)

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoded_columns
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoded_columns
)

# Combine numeric + encoded categorical features
X_train_final = pd.concat(
    [X_train_numeric, X_train_encoded],
    axis=1
)

X_test_final = pd.concat(
    [X_test_numeric, X_test_encoded],
    axis=1
)

print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape:", X_test_final.shape)

print("\nCategorical encoded columns:")
print(encoded_columns.tolist())

print("\nMissing values:")
print("Train:", X_train_final.isna().sum().sum())
print("Test:", X_test_final.isna().sum().sum())

X_train_final shape: (809515, 50)
X_test_final shape: (190412, 50)

Categorical encoded columns:
['candidate_room_type_Conference', 'candidate_room_type_Focus', 'candidate_room_type_Meeting', 'candidate_room_type_Training', 'requested_room_type_Conference', 'requested_room_type_Focus', 'requested_room_type_Meeting', 'requested_room_type_Training']

Missing values:
Train: 0
Test: 0


In [42]:
#  BASELINE MODEL

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

baseline_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

print("Training baseline model...")

baseline_model.fit(
    X_train_final,
    y_train
)

print("Training complete.")

# Probability of positive candidate
y_test_proba = baseline_model.predict_proba(
    X_test_final
)[:, 1]

# Default classification threshold
y_test_pred = (
    y_test_proba >= 0.5
).astype(int)

print("\n========== BASELINE RESULTS ==========")

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, y_test_proba), 4)
)

print(
    "PR-AUC:",
    round(average_precision_score(y_test, y_test_proba), 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        digits=4
    )
)

Training baseline model...


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training complete.

========== BASELINE RESULTS ==========
ROC-AUC: 0.9724
PR-AUC: 0.5883

Classification Report:
              precision    recall  f1-score   support

           0     0.9977    0.8997    0.9462    183923
           1     0.2489    0.9421    0.3938      6489

    accuracy                         0.9012    190412
   macro avg     0.6233    0.9209    0.6700    190412
weighted avg     0.9722    0.9012    0.9274    190412



In [ ]:
# =========================================================================================================
#       LOGISTIC REGRESSION MODEL - TRAINING
# =========================================================================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

scaled_baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

print("Training scaled baseline model...")

scaled_baseline_model.fit(
    X_train_final,
    y_train
)

print("Training complete.")

y_test_proba_scaled = scaled_baseline_model.predict_proba(
    X_test_final
)[:, 1]

y_test_pred_scaled = (
    y_test_proba_scaled >= 0.5
).astype(int)

print("\n========== SCALED BASELINE RESULTS ==========")

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, y_test_proba_scaled), 4)
)

print(
    "PR-AUC:",
    round(average_precision_score(y_test, y_test_proba_scaled), 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred_scaled,
        digits=4
    )
)

Training scaled baseline model...
Training complete.

========== SCALED BASELINE RESULTS ==========
ROC-AUC: 0.9725
PR-AUC: 0.5911

Classification Report:
              precision    recall  f1-score   support

           0     0.9978    0.8998    0.9462    183923
           1     0.2492    0.9428    0.3942      6489

    accuracy                         0.9012    190412
   macro avg     0.6235    0.9213    0.6702    190412
weighted avg     0.9723    0.9012    0.9274    190412



In [44]:
# =========================================================================================================
#       RANDOM FOREST - TRAINING
# =========================================================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

random_forest_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")

random_forest_model.fit(
    X_train_final,
    y_train
)

print("Training complete.")

y_test_proba_rf = random_forest_model.predict_proba(
    X_test_final
)[:, 1]

y_test_pred_rf = (
    y_test_proba_rf >= 0.5
).astype(int)

print("\n========== RANDOM FOREST RESULTS ==========")

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, y_test_proba_rf), 4)
)

print(
    "PR-AUC:",
    round(average_precision_score(y_test, y_test_proba_rf), 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred_rf,
        digits=4
    )
)

Training Random Forest...
Training complete.

========== RANDOM FOREST RESULTS ==========
ROC-AUC: 0.9932
PR-AUC: 0.8585

Classification Report:
              precision    recall  f1-score   support

           0     0.9987    0.9611    0.9795    183923
           1     0.4666    0.9649    0.6290      6489

    accuracy                         0.9612    190412
   macro avg     0.7327    0.9630    0.8043    190412
weighted avg     0.9806    0.9612    0.9676    190412



In [45]:
# =========================================================================================================
#       RANDOM FOREST - TRAINING  --------------DEEPLY VALIDATING RANDOM FOREST IN RANKING ALSO
# =========================================================================================================

# Attach Random Forest scores to the test candidates
ranking_data = test_data[
    ["booking_id", "candidate_room_id", "candidate_start_datetime", "target"]
].copy()

ranking_data["model_score"] = y_test_proba_rf

# Rank candidates within each booking
ranking_data["rank"] = (
    ranking_data
    .groupby("booking_id")["model_score"]
    .rank(
        method="first",
        ascending=False
    )
)

# For each request, check where the actual historical choice ranked
positive_ranks = (
    ranking_data[ranking_data["target"] == 1]
    [["booking_id", "rank"]]
    .copy()
)

# Top-K hit rates
total_requests = len(positive_ranks)

top_1_hits = (positive_ranks["rank"] <= 1).sum()
top_3_hits = (positive_ranks["rank"] <= 3).sum()
top_5_hits = (positive_ranks["rank"] <= 5).sum()
top_10_hits = (positive_ranks["rank"] <= 10).sum()

print("========== RANKING PERFORMANCE ==========")

print(
    f"Top-1 Accuracy:  {top_1_hits / total_requests:.4f} "
    f"({top_1_hits:,}/{total_requests:,})"
)

print(
    f"Top-3 Accuracy:  {top_3_hits / total_requests:.4f} "
    f"({top_3_hits:,}/{total_requests:,})"
)

print(
    f"Top-5 Accuracy:  {top_5_hits / total_requests:.4f} "
    f"({top_5_hits:,}/{total_requests:,})"
)

print(
    f"Top-10 Accuracy: {top_10_hits / total_requests:.4f} "
    f"({top_10_hits:,}/{total_requests:,})"
)

print("\nActual positive candidate rank:")
print(positive_ranks["rank"].describe())

========== RANKING PERFORMANCE ==========
Top-1 Accuracy:  0.9841 (6,386/6,489)
Top-3 Accuracy:  0.9980 (6,476/6,489)
Top-5 Accuracy:  0.9991 (6,483/6,489)
Top-10 Accuracy: 1.0000 (6,489/6,489)

Actual positive candidate rank:
count    6489.000000
mean        1.026352
std         0.272703
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        10.000000
Name: rank, dtype: float64


In [46]:
# ====================================================================================================================================
#       ITNI HIGH ACCURACY KESE ARHI HAI (98%) , LETS CHECK KAUNSE FEATURES EXTRACT KARKE ITNA ACHHA PREDICT KAR RHA HAI
# ===================================================================================================================================

feature_importance = pd.DataFrame({
    "feature": X_train_final.columns,
    "importance": random_forest_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("========== TOP 20 FEATURE IMPORTANCES ==========")
display(feature_importance.head(20))

========== TOP 20 FEATURE IMPORTANCES ==========


,feature,importance
0,start_time_difference_minutes,0.242020
1,flexibility_used_pct,0.194940
2,candidate_start_hour,0.156140
3,requested_capacity,0.090964
4,max_time_flexibility_minutes,0.050078
5,candidate_capacity,0.048955
6,room_type_match,0.045268
7,requested_start_hour,0.036781
8,floor_difference,0.021778
9,requested_day_of_week,0.012808


In [47]:
def get_feature_group(feature):
    # Employee historical behaviour
    if (
        "requested_room_type_" in feature and feature.endswith("_pct")
    ) or (
        "requested_floor_" in feature and feature.endswith("_pct")
    ) or (
        "requested_start_hour_" in feature and feature.endswith("_pct")
    ) or (
        "requested_capacity_" in feature and feature.endswith("_pct")
    ) or feature in [
        "duration_mean",
        "duration_std",
        "flexibility_mean",
        "flexibility_std",
        "requested_room_type_shift_pct",
        "requested_floor_shift_pct",
        "requested_start_hour_shift_pct",
        "requested_capacity_shift_pct"
    ]:
        return "Employee historical behaviour"

    # Current request
    if feature in [
        "requested_room_type",
        "requested_floor",
        "requested_capacity",
        "requested_duration_minutes",
        "max_time_flexibility_minutes",
        "requested_start_hour",
        "requested_day_of_week"
    ]:
        return "Current request"

    # Candidate room
    if feature in [
        "candidate_floor",
        "candidate_room_type",
        "candidate_capacity",
        "candidate_start_hour"
    ]:
        return "Candidate room/time"

    # Employee-candidate relationship
    if feature in [
        "room_type_match",
        "floor_difference",
        "capacity_difference",
        "start_time_difference_minutes",
        "flexibility_used_pct"
    ]:
        return "Employee-candidate relationship"

    return "Other"


feature_importance["feature_group"] = (
    feature_importance["feature"]
    .apply(get_feature_group)
)

grouped_importance = (
    feature_importance
    .groupby("feature_group")["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

grouped_importance["importance_pct"] = (
    grouped_importance["importance"] * 100
)

print("========== FEATURE IMPORTANCE BY GROUP ==========")
display(grouped_importance)

========== FEATURE IMPORTANCE BY GROUP ==========


,feature_group,importance,importance_pct
0,Employee-candidate relationship,0.514019,51.401942
1,Candidate room/time,0.208574,20.857390
2,Current request,0.196879,19.687890
3,Employee historical behaviour,0.051856,5.185600
4,Other,0.028672,2.867178


In [48]:
def get_feature_group(feature):

    # Employee historical behaviour
    if (
        feature.startswith("requested_room_type_") and feature.endswith("_pct")
        or feature.startswith("requested_floor_") and feature.endswith("_pct")
        or feature.startswith("requested_start_hour_") and feature.endswith("_pct")
        or feature.startswith("requested_capacity_") and feature.endswith("_pct")
        or feature in [
            "duration_mean",
            "duration_std",
            "flexibility_mean",
            "flexibility_std",
            "requested_room_type_shift_pct",
            "requested_floor_shift_pct",
            "requested_start_hour_shift_pct",
            "requested_capacity_shift_pct"
        ]
    ):
        return "Employee historical behaviour"

    # Current request
    if (
        feature in [
            "requested_floor",
            "requested_capacity",
            "requested_duration_minutes",
            "max_time_flexibility_minutes",
            "requested_start_hour",
            "requested_day_of_week"
        ]
        or feature.startswith("requested_room_type_")
        and not feature.endswith("_pct")
    ):
        return "Current request"

    # Candidate room/time
    if (
        feature in [
            "candidate_floor",
            "candidate_capacity",
            "candidate_start_hour"
        ]
        or feature.startswith("candidate_room_type_")
    ):
        return "Candidate room/time"

    # Employee-candidate relationship
    if feature in [
        "room_type_match",
        "floor_difference",
        "capacity_difference",
        "start_time_difference_minutes",
        "flexibility_used_pct"
    ]:
        return "Employee-candidate relationship"

    return "Other"


feature_importance["feature_group"] = (
    feature_importance["feature"]
    .apply(get_feature_group)
)

grouped_importance = (
    feature_importance
    .groupby("feature_group")["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

grouped_importance["importance_pct"] = (
    grouped_importance["importance"] * 100
)

print("========== CORRECTED FEATURE IMPORTANCE BY GROUP ==========")
display(grouped_importance)

========== CORRECTED FEATURE IMPORTANCE BY GROUP ==========


,feature_group,importance,importance_pct
0,Employee-candidate relationship,0.514019,51.401942
1,Candidate room/time,0.229955,22.995532
2,Current request,0.204169,20.416926
3,Employee historical behaviour,0.051856,5.185600


In [49]:
# ====================================================================================================================================
#       Random Forest with employee historical behaviour vs Random Forest without employee historical behaviou
# ===================================================================================================================================


# Identify employee historical behaviour features
historical_behavior_features = [
    col for col in X_train_final.columns
    if (
        col.endswith("_pct")
        and (
            col.startswith("requested_room_type_")
            or col.startswith("requested_floor_")
            or col.startswith("requested_start_hour_")
            or col.startswith("requested_capacity_")
        )
    )
    or col in [
        "duration_mean",
        "duration_std",
        "flexibility_mean",
        "flexibility_std",
        "requested_room_type_shift_pct",
        "requested_floor_shift_pct",
        "requested_start_hour_shift_pct",
        "requested_capacity_shift_pct"
    ]
]

print("Historical behaviour features removed:", len(historical_behavior_features))
print(historical_behavior_features)

# Remove historical behaviour features
X_train_no_history = X_train_final.drop(
    columns=historical_behavior_features
)

X_test_no_history = X_test_final.drop(
    columns=historical_behavior_features
)

print("\nFeatures with history:", X_train_final.shape[1])
print("Features without history:", X_train_no_history.shape[1])

# Same Random Forest configuration as our main model
rf_no_history = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Random Forest WITHOUT employee history...")

rf_no_history.fit(
    X_train_no_history,
    y_train
)

print("Training complete.")

# Predictions
y_test_proba_no_history = rf_no_history.predict_proba(
    X_test_no_history
)[:, 1]

# Classification metrics
print("\n========== WITHOUT HISTORY ==========")

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, y_test_proba_no_history), 4)
)

print(
    "PR-AUC:",
    round(
        average_precision_score(
            y_test,
            y_test_proba_no_history
        ),
        4
    )
)

# Ranking evaluation
ranking_no_history = test_data[
    [
        "booking_id",
        "candidate_room_id",
        "candidate_start_datetime",
        "target"
    ]
].copy()

ranking_no_history["model_score"] = y_test_proba_no_history

ranking_no_history["rank"] = (
    ranking_no_history
    .groupby("booking_id")["model_score"]
    .rank(
        method="first",
        ascending=False
    )
)

positive_ranks_no_history = ranking_no_history[
    ranking_no_history["target"] == 1
][
    ["booking_id", "rank"]
]

total_test_requests = len(positive_ranks_no_history)

for k in [1, 3, 5, 10]:
    hits = (
        positive_ranks_no_history["rank"] <= k
    ).sum()

    print(
        f"Top-{k} Accuracy:",
        round(hits / total_test_requests, 4),
        f"({hits:,}/{total_test_requests:,})"
    )

Historical behaviour features removed: 28
['requested_room_type_Conference_pct', 'requested_room_type_Focus_pct', 'requested_room_type_Meeting_pct', 'requested_room_type_Training_pct', 'requested_floor_1_pct', 'requested_floor_2_pct', 'requested_floor_3_pct', 'requested_floor_4_pct', 'requested_floor_5_pct', 'requested_start_hour_8_pct', 'requested_start_hour_9_pct', 'requested_start_hour_10_pct', 'requested_start_hour_11_pct', 'requested_capacity_2_pct', 'requested_capacity_4_pct', 'requested_capacity_6_pct', 'requested_capacity_8_pct', 'requested_capacity_10_pct', 'requested_capacity_12_pct', 'requested_capacity_16_pct', 'duration_mean', 'duration_std', 'flexibility_mean', 'flexibility_std', 'requested_room_type_shift_pct', 'requested_floor_shift_pct', 'requested_start_hour_shift_pct', 'requested_capacity_shift_pct']

Features with history: 50
Features without history: 22

Training Random Forest WITHOUT employee history...
Training complete.

========== WITHOUT HISTORY ==========
ROC

In [50]:
preference_rows = []

for _, request in allocated_requests.iterrows():

    employee_id = request["employee_id"]
    current_time = request["requested_start_datetime"]

    # Only history BEFORE the current request
    employee_history = all_requests[
        (all_requests["employee_id"] == employee_id) &
        (all_requests["requested_start_datetime"] < current_time)
    ]

    # Last 30 days of history before current request
    recent_start = current_time - pd.Timedelta(days=30)

    recent_history = employee_history[
        employee_history["requested_start_datetime"] >= recent_start
    ]

    row = {
        "booking_id": request["booking_id"],
        "employee_id": employee_id
    }

    # ---------- LONG-TERM PREFERENCES ----------
    for column, values in behavior_columns.items():

        counts = employee_history[column].value_counts()
        total = len(employee_history)

        for value in values:
            row[f"lt_{column}_{value}_pct"] = (
                counts.get(value, 0) / total * 100
                if total > 0 else 0.0
            )

    # ---------- RECENT PREFERENCES ----------
    for column, values in behavior_columns.items():

        counts = recent_history[column].value_counts()
        total = len(recent_history)

        for value in values:
            row[f"recent_{column}_{value}_pct"] = (
                counts.get(value, 0) / total * 100
                if total > 0 else 0.0
            )

    preference_rows.append(row)


request_preference_profiles = pd.DataFrame(preference_rows)

print("Preference profile shape:", request_preference_profiles.shape)
print("Missing values:", request_preference_profiles.isna().sum().sum())

print("\nFirst 15 columns:")
print(request_preference_profiles.columns.tolist()[:15])

print("\nSample:")
display(request_preference_profiles.head())

Preference profile shape: (32442, 42)
Missing values: 0

First 15 columns:
['booking_id', 'employee_id', 'lt_requested_room_type_Conference_pct', 'lt_requested_room_type_Focus_pct', 'lt_requested_room_type_Meeting_pct', 'lt_requested_room_type_Training_pct', 'lt_requested_floor_1_pct', 'lt_requested_floor_2_pct', 'lt_requested_floor_3_pct', 'lt_requested_floor_4_pct', 'lt_requested_floor_5_pct', 'lt_requested_start_hour_8_pct', 'lt_requested_start_hour_9_pct', 'lt_requested_start_hour_10_pct', 'lt_requested_start_hour_11_pct']

Sample:


,booking_id,employee_id,lt_requested_room_type_Conference_pct,lt_requested_room_type_Focus_pct,lt_requested_room_type_Meeting_pct,lt_requested_room_type_Training_pct,lt_requested_floor_1_pct,lt_requested_floor_2_pct,lt_requested_floor_3_pct,lt_requested_floor_4_pct,...,recent_requested_start_hour_9_pct,recent_requested_start_hour_10_pct,recent_requested_start_hour_11_pct,recent_requested_capacity_2_pct,recent_requested_capacity_4_pct,recent_requested_capacity_6_pct,recent_requested_capacity_8_pct,recent_requested_capacity_10_pct,recent_requested_capacity_12_pct,recent_requested_capacity_16_pct
0,RB06430,E0742,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,RB10999,E0166,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,RB19847,E0605,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,RB22582,E0904,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,RB25100,E0378,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [51]:
# Merge clean long-term/recent preference profiles
ml_dataset = ml_dataset.merge(
    request_preference_profiles,
    on=["booking_id", "employee_id"],
    how="left"
)

# --------------------------------------------------
# 1. Candidate room-type preference
# --------------------------------------------------
room_type_map = {
    "Conference": "Conference",
    "Focus": "Focus",
    "Meeting": "Meeting",
    "Training": "Training"
}

ml_dataset["candidate_room_type_lt_preference_pct"] = (
    ml_dataset.apply(
        lambda row: row[
            f"lt_requested_room_type_{room_type_map[row['candidate_room_type']]}_pct"
        ],
        axis=1
    )
)

ml_dataset["candidate_room_type_recent_preference_pct"] = (
    ml_dataset.apply(
        lambda row: row[
            f"recent_requested_room_type_{room_type_map[row['candidate_room_type']]}_pct"
        ],
        axis=1
    )
)


# --------------------------------------------------
# 2. Candidate floor preference
# --------------------------------------------------
ml_dataset["candidate_floor_lt_preference_pct"] = (
    ml_dataset.apply(
        lambda row: row[
            f"lt_requested_floor_{int(row['candidate_floor'])}_pct"
        ],
        axis=1
    )
)

ml_dataset["candidate_floor_recent_preference_pct"] = (
    ml_dataset.apply(
        lambda row: row[
            f"recent_requested_floor_{int(row['candidate_floor'])}_pct"
        ],
        axis=1
    )
)


# --------------------------------------------------
# 3. Candidate start-hour preference
# --------------------------------------------------
ml_dataset["candidate_start_hour_lt_preference_pct"] = (
    ml_dataset.apply(
        lambda row: row[
            f"lt_requested_start_hour_{int(row['candidate_start_hour'])}_pct"
        ],
        axis=1
    )
)

ml_dataset["candidate_start_hour_recent_preference_pct"] = (
    ml_dataset.apply(
        lambda row: row[
            f"recent_requested_start_hour_{int(row['candidate_start_hour'])}_pct"
        ],
        axis=1
    )
)


# --------------------------------------------------
# 4. Candidate capacity preference
# --------------------------------------------------
ml_dataset["candidate_capacity_lt_preference_pct"] = (
    ml_dataset.apply(
        lambda row: row[
            f"lt_requested_capacity_{int(row['candidate_capacity'])}_pct"
        ],
        axis=1
    )
)

ml_dataset["candidate_capacity_recent_preference_pct"] = (
    ml_dataset.apply(
        lambda row: row[
            f"recent_requested_capacity_{int(row['candidate_capacity'])}_pct"
        ],
        axis=1
    )
)


print("ML dataset shape:", ml_dataset.shape)

print("\nMissing values:")
print(ml_dataset.isna().sum().sum())

print("\nNew personalization features:")
print([
    "candidate_room_type_lt_preference_pct",
    "candidate_room_type_recent_preference_pct",
    "candidate_floor_lt_preference_pct",
    "candidate_floor_recent_preference_pct",
    "candidate_start_hour_lt_preference_pct",
    "candidate_start_hour_recent_preference_pct",
    "candidate_capacity_lt_preference_pct",
    "candidate_capacity_recent_preference_pct"
])

KeyError: 'lt_requested_start_hour_7_pct'

In [52]:
print("Candidate start hours:")
print(
    sorted(
        ml_dataset["candidate_start_hour"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nHistorical start-hour preference columns:")
print([
    col for col in request_preference_profiles.columns
    if "requested_start_hour" in col
])

Candidate start hours:
[6, 7, 8, 9, 10, 11, 12, 13]

Historical start-hour preference columns:
['lt_requested_start_hour_8_pct', 'lt_requested_start_hour_9_pct', 'lt_requested_start_hour_10_pct', 'lt_requested_start_hour_11_pct', 'recent_requested_start_hour_8_pct', 'recent_requested_start_hour_9_pct', 'recent_requested_start_hour_10_pct', 'recent_requested_start_hour_11_pct']


In [53]:
# --------------------------------------------------
# 1. Candidate room-type preference
# --------------------------------------------------
def get_preference(row, prefix, dimension, value):
    column = f"{prefix}_{dimension}_{value}_pct"

    if column in row.index:
        return row[column]

    return 0.0


ml_dataset["candidate_room_type_lt_preference_pct"] = (
    ml_dataset.apply(
        lambda row: get_preference(
            row,
            "lt",
            "requested_room_type",
            row["candidate_room_type"]
        ),
        axis=1
    )
)

ml_dataset["candidate_room_type_recent_preference_pct"] = (
    ml_dataset.apply(
        lambda row: get_preference(
            row,
            "recent",
            "requested_room_type",
            row["candidate_room_type"]
        ),
        axis=1
    )
)


# --------------------------------------------------
# 2. Candidate floor preference
# --------------------------------------------------
ml_dataset["candidate_floor_lt_preference_pct"] = (
    ml_dataset.apply(
        lambda row: get_preference(
            row,
            "lt",
            "requested_floor",
            int(row["candidate_floor"])
        ),
        axis=1
    )
)

ml_dataset["candidate_floor_recent_preference_pct"] = (
    ml_dataset.apply(
        lambda row: get_preference(
            row,
            "recent",
            "requested_floor",
            int(row["candidate_floor"])
        ),
        axis=1
    )
)


# --------------------------------------------------
# 3. Candidate start-hour preference
# --------------------------------------------------
ml_dataset["candidate_start_hour_lt_preference_pct"] = (
    ml_dataset.apply(
        lambda row: get_preference(
            row,
            "lt",
            "requested_start_hour",
            int(row["candidate_start_hour"])
        ),
        axis=1
    )
)

ml_dataset["candidate_start_hour_recent_preference_pct"] = (
    ml_dataset.apply(
        lambda row: get_preference(
            row,
            "recent",
            "requested_start_hour",
            int(row["candidate_start_hour"])
        ),
        axis=1
    )
)


# --------------------------------------------------
# 4. Candidate capacity preference
# --------------------------------------------------
ml_dataset["candidate_capacity_lt_preference_pct"] = (
    ml_dataset.apply(
        lambda row: get_preference(
            row,
            "lt",
            "requested_capacity",
            int(row["candidate_capacity"])
        ),
        axis=1
    )
)

ml_dataset["candidate_capacity_recent_preference_pct"] = (
    ml_dataset.apply(
        lambda row: get_preference(
            row,
            "recent",
            "requested_capacity",
            int(row["candidate_capacity"])
        ),
        axis=1
    )
)


print("ML dataset shape:", ml_dataset.shape)
print("Missing values:", ml_dataset.isna().sum().sum())

print("\nPersonalization feature summary:")
display(
    ml_dataset[
        [
            "candidate_room_type_lt_preference_pct",
            "candidate_room_type_recent_preference_pct",
            "candidate_floor_lt_preference_pct",
            "candidate_floor_recent_preference_pct",
            "candidate_start_hour_lt_preference_pct",
            "candidate_start_hour_recent_preference_pct",
            "candidate_capacity_lt_preference_pct",
            "candidate_capacity_recent_preference_pct"
        ]
    ].describe()
)

ML dataset shape: (999927, 99)
Missing values: 0

Personalization feature summary:


,candidate_room_type_lt_preference_pct,candidate_room_type_recent_preference_pct,candidate_floor_lt_preference_pct,candidate_floor_recent_preference_pct,candidate_start_hour_lt_preference_pct,candidate_start_hour_recent_preference_pct,candidate_capacity_lt_preference_pct,candidate_capacity_recent_preference_pct
count,999927.000000,999927.000000,999927.000000,999927.000000,999927.000000,999927.000000,999927.000000,999927.000000
mean,20.815198,19.978092,20.400571,19.626497,17.125168,16.551028,24.096124,23.326142
std,34.076185,35.429024,34.012766,35.367007,28.337124,30.796609,23.380080,29.862189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.724138,0.000000
50%,4.166667,0.000000,3.225806,0.000000,0.000000,0.000000,20.000000,0.000000
75%,14.285714,20.000000,13.636364,20.000000,20.000000,20.000000,42.857143,42.857143
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [54]:
personalization_columns = [
    "candidate_room_type_lt_preference_pct",
    "candidate_room_type_recent_preference_pct",
    "candidate_floor_lt_preference_pct",
    "candidate_floor_recent_preference_pct",
    "candidate_start_hour_lt_preference_pct",
    "candidate_start_hour_recent_preference_pct",
    "candidate_capacity_lt_preference_pct",
    "candidate_capacity_recent_preference_pct"
]

# How many unique values each personalization feature has
print("Unique values:")
print(
    ml_dataset[personalization_columns]
    .nunique()
)

# Check one employee's candidates
sample_employee = ml_dataset["employee_id"].iloc[0]

sample_candidates = (
    ml_dataset[
        ml_dataset["employee_id"] == sample_employee
    ][
        [
            "booking_id",
            "candidate_room_id",
            "candidate_room_type",
            "candidate_floor",
            "candidate_start_hour",
            "candidate_capacity"
        ] + personalization_columns
    ]
    .head(10)
)

print("\nSample employee:", sample_employee)
display(sample_candidates)

Unique values:
candidate_room_type_lt_preference_pct         1162
candidate_room_type_recent_preference_pct       67
candidate_floor_lt_preference_pct             1131
candidate_floor_recent_preference_pct           70
candidate_start_hour_lt_preference_pct        1330
candidate_start_hour_recent_preference_pct      75
candidate_capacity_lt_preference_pct          1281
candidate_capacity_recent_preference_pct        65
dtype: int64

Sample employee: E0742


,booking_id,candidate_room_id,candidate_room_type,candidate_floor,candidate_start_hour,candidate_capacity,candidate_room_type_lt_preference_pct,candidate_room_type_recent_preference_pct,candidate_floor_lt_preference_pct,candidate_floor_recent_preference_pct,candidate_start_hour_lt_preference_pct,candidate_start_hour_recent_preference_pct,candidate_capacity_lt_preference_pct,candidate_capacity_recent_preference_pct
0,RB06430,R005,Training,1,7,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,RB06430,R006,Focus,1,7,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,RB06430,R008,Meeting,1,7,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,RB06430,R008,Meeting,1,7,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,RB06430,R008,Meeting,1,7,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,RB06430,R008,Meeting,1,7,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,RB06430,R008,Meeting,1,8,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,RB06430,R040,Training,4,7,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
63423,RB14000,R022,Focus,3,8,4,0.0,0.0,100.0,100.0,100.0,100.0,0.0,0.0
63424,RB14000,R022,Focus,3,9,4,0.0,0.0,100.0,100.0,0.0,0.0,0.0,0.0


In [55]:
# Combine long-term and recent preference into one score.
# Recent behaviour gets slightly more weight because it reflects
# the employee's current preference more strongly.

ml_dataset["room_type_personalization_score"] = (
    0.6 * ml_dataset["candidate_room_type_recent_preference_pct"]
    + 0.4 * ml_dataset["candidate_room_type_lt_preference_pct"]
)

ml_dataset["floor_personalization_score"] = (
    0.6 * ml_dataset["candidate_floor_recent_preference_pct"]
    + 0.4 * ml_dataset["candidate_floor_lt_preference_pct"]
)

ml_dataset["start_hour_personalization_score"] = (
    0.6 * ml_dataset["candidate_start_hour_recent_preference_pct"]
    + 0.4 * ml_dataset["candidate_start_hour_lt_preference_pct"]
)

ml_dataset["capacity_personalization_score"] = (
    0.6 * ml_dataset["candidate_capacity_recent_preference_pct"]
    + 0.4 * ml_dataset["candidate_capacity_lt_preference_pct"]
)

personalization_score_columns = [
    "room_type_personalization_score",
    "floor_personalization_score",
    "start_hour_personalization_score",
    "capacity_personalization_score"
]

print("Personalization scores created:", len(personalization_score_columns))

print("\nMissing values:")
print(
    ml_dataset[personalization_score_columns]
    .isna()
    .sum()
    .sum()
)

print("\nPersonalization score summary:")
display(
    ml_dataset[personalization_score_columns]
    .describe()
)

Personalization scores created: 4

Missing values:
0

Personalization score summary:


,room_type_personalization_score,floor_personalization_score,start_hour_personalization_score,capacity_personalization_score
count,999927.000000,999927.000000,999927.000000,999927.000000
mean,20.312934,19.936126,16.780684,23.634135
std,34.222342,34.167711,28.963716,25.612213
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.701754
50%,1.904762,1.379310,0.000000,15.000000
75%,19.210526,18.451613,21.153846,41.333333
max,100.000000,100.000000,100.000000,100.000000


In [56]:
# Add the 4 candidate-specific personalization scores
personalization_features = [
    "room_type_personalization_score",
    "floor_personalization_score",
    "start_hour_personalization_score",
    "capacity_personalization_score"
]

# Create training/test feature sets from the existing final features
X_train_personalized = X_train_final.copy()
X_test_personalized = X_test_final.copy()

# Add personalization features using booking_id alignment
train_personalization = (
    ml_dataset[
        ml_dataset["booking_id"].isin(train_booking_ids)
    ][["booking_id"] + personalization_features]
    .drop_duplicates("booking_id")
)

test_personalization = (
    ml_dataset[
        ml_dataset["booking_id"].isin(test_booking_ids)
    ][["booking_id"] + personalization_features]
    .drop_duplicates("booking_id")
)

# Align personalization values with train/test rows
X_train_personalized = (
    train_data[["booking_id"]]
    .merge(train_personalization, on="booking_id", how="left")
    [personalization_features]
    .reset_index(drop=True)
)

X_test_personalized = (
    test_data[["booking_id"]]
    .merge(test_personalization, on="booking_id", how="left")
    [personalization_features]
    .reset_index(drop=True)
)

# Add the original 50 features
X_train_personalized = pd.concat(
    [
        X_train_final.reset_index(drop=True),
        X_train_personalized
    ],
    axis=1
)

X_test_personalized = pd.concat(
    [
        X_test_final.reset_index(drop=True),
        X_test_personalized
    ],
    axis=1
)

print("Personalized X_train shape:", X_train_personalized.shape)
print("Personalized X_test shape:", X_test_personalized.shape)
print(
    "Missing values:",
    X_train_personalized.isna().sum().sum(),
    X_test_personalized.isna().sum().sum()
)

# Train the same Random Forest configuration
rf_personalized = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nTraining PERSONALIZED Random Forest...")

rf_personalized.fit(
    X_train_personalized,
    y_train
)

print("Training complete.")

# Test probabilities
y_test_proba_personalized = (
    rf_personalized
    .predict_proba(X_test_personalized)[:, 1]
)

print("\n========== PERSONALIZED RANDOM FOREST ==========")

print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            y_test,
            y_test_proba_personalized
        ),
        4
    )
)

print(
    "PR-AUC:",
    round(
        average_precision_score(
            y_test,
            y_test_proba_personalized
        ),
        4
    )
)

Personalized X_train shape: (809515, 54)
Personalized X_test shape: (190412, 54)
Missing values: 0 0

Training PERSONALIZED Random Forest...
Training complete.

========== PERSONALIZED RANDOM FOREST ==========
ROC-AUC: 0.9935
PR-AUC: 0.8662


In [ ]:
# ======================================================================================================================================
# =======================================================================================================================================
# Personalized Random Forest training after adding candidate specific percentages too
# ======================================================================================================================================
# =======================================================================================================================================


# Create ranking dataset for the personalized model
ranking_personalized = test_data[
    [
        "booking_id",
        "candidate_room_id",
        "candidate_start_datetime",
        "target"
    ]
].copy()

ranking_personalized["model_score"] = y_test_proba_personalized

# Rank candidates within each booking
ranking_personalized["rank"] = (
    ranking_personalized
    .groupby("booking_id")["model_score"]
    .rank(
        method="first",
        ascending=False
    )
)

# Keep the actual historically selected candidate
positive_ranks_personalized = ranking_personalized[
    ranking_personalized["target"] == 1
][
    ["booking_id", "rank"]
]

total_requests = len(positive_ranks_personalized)

print("========== PERSONALIZED RANKING PERFORMANCE ==========")

for k in [1, 3, 5, 10]:
    hits = (
        positive_ranks_personalized["rank"] <= k
    ).sum()

    print(
        f"Top-{k} Accuracy:",
        round(hits / total_requests, 4),
        f"({hits:,}/{total_requests:,})"
    )

print("\nActual positive candidate rank:")
print(
    positive_ranks_personalized["rank"].describe()
)

========== PERSONALIZED RANKING PERFORMANCE ==========
Top-1 Accuracy: 0.983 (6,379/6,489)
Top-3 Accuracy: 0.9977 (6,474/6,489)
Top-5 Accuracy: 0.9991 (6,483/6,489)
Top-10 Accuracy: 1.0 (6,489/6,489)

Actual positive candidate rank:
count    6489.000000
mean        1.028664
std         0.288141
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        10.000000
Name: rank, dtype: float64


In [58]:
personalized_feature_importance = pd.DataFrame({
    "feature": X_train_personalized.columns,
    "importance": rf_personalized.feature_importances_
})

personalized_feature_importance = (
    personalized_feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("========== TOP 20 PERSONALIZED FEATURE IMPORTANCES ==========")
display(
    personalized_feature_importance.head(20)
)

print("\n========== PERSONALIZATION FEATURES ==========")

display(
    personalized_feature_importance[
        personalized_feature_importance["feature"].isin(
            personalization_features
        )
    ]
)

========== TOP 20 PERSONALIZED FEATURE IMPORTANCES ==========


,feature,importance
0,start_time_difference_minutes,0.220165
1,flexibility_used_pct,0.191198
2,candidate_start_hour,0.138774
3,requested_capacity,0.089105
4,room_type_match,0.057914
5,max_time_flexibility_minutes,0.048572
6,candidate_capacity,0.044628
7,start_hour_personalization_score,0.033623
8,requested_start_hour,0.032043
9,floor_difference,0.020555



========== PERSONALIZATION FEATURES ==========


,feature,importance
7,start_hour_personalization_score,0.033623
13,capacity_personalization_score,0.008955
15,room_type_personalization_score,0.005144
19,floor_personalization_score,0.003343
